In [1]:
!nvidia-smi

Thu Jul  9 15:27:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics supervision opencv-python pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 10.9 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO
import supervision as sv
import cv2
import pandas as pd
import numpy as np

print("All libraries installed successfully")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All libraries installed successfully


In [4]:
from google.colab import files

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

print(video_path)

Saving 1000182937.mp4 to 1000182937.mp4
1000182937.mp4


In [6]:
# Install (run once)
!pip install -q ultralytics supervision opencv-python pandas

from ultralytics import YOLO
import supervision as sv
import cv2
import numpy as np
import pandas as pd
import os
from google.colab import files

# Upload video
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Load model
model = YOLO("yolov8m.pt")

# Open video
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

# Output video
out = cv2.VideoWriter(
    "crowd_panic_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# Tracker
tracker = sv.ByteTrack()

previous_positions = {}

os.makedirs("snapshots", exist_ok=True)

records = []
frame_no = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, verbose=False)[0]

    detections = sv.Detections.from_ultralytics(results)

    # Keep only people
    if detections.class_id is not None:
        mask = detections.class_id == 0
        detections = detections[mask]

    detections = tracker.update_with_detections(detections)

    panic_count = 0
    warning_count = 0

    if detections.tracker_id is not None:

        for box, tid in zip(
            detections.xyxy,
            detections.tracker_id
        ):

            x1, y1, x2, y2 = box

            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            speed = 0

            if tid in previous_positions:

                px, py = previous_positions[tid]

                speed = np.sqrt(
                    (cx - px) ** 2 +
                    (cy - py) ** 2
                )

            previous_positions[tid] = (cx, cy)

            # PERSON STATUS

            if speed < 8:
                state = "NORMAL"
                color = (0, 255, 0)      # Green

            elif speed < 20:
                state = "WARNING"
                color = (0, 255, 255)    # Yellow
                warning_count += 1

            else:
                state = "PANIC"
                color = (0, 0, 255)      # Red
                panic_count += 1

            cv2.rectangle(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                color,
                2
            )

            cv2.putText(
                frame,
                state,
                (int(x1), int(y1) - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    people = len(detections)

    # CROWD STATUS

    if panic_count >= 5:

        crowd_status = "PANIC ALERT"
        status_color = (0, 0, 255)

        cv2.imwrite(
            f"snapshots/panic_{frame_no}.jpg",
            frame
        )

    elif warning_count >= 5:

        crowd_status = "WARNING"
        status_color = (0, 255, 255)

    else:

        crowd_status = "NORMAL"
        status_color = (0, 255, 0)

    cv2.putText(
        frame,
        crowd_status,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        status_color,
        3
    )

    cv2.putText(
        frame,
        f"People: {people}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    records.append([
        frame_no,
        people,
        warning_count,
        panic_count,
        crowd_status
    ])

    out.write(frame)

    frame_no += 1

cap.release()
out.release()

# Save report
df = pd.DataFrame(
    records,
    columns=[
        "Frame",
        "People",
        "Warning_Count",
        "Panic_Count",
        "Status"
    ]
)

df.to_csv("crowd_report.csv", index=False)

print("DONE")
print("Video saved as crowd_panic_output.mp4")
print("CSV saved as crowd_report.csv")

Saving 1000182937.mp4 to 1000182937 (1).mp4


DONE
Video saved as crowd_panic_output.mp4
CSV saved as crowd_report.csv


In [7]:
from google.colab import files

files.download("crowd_panic_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
files.download("crowd_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import cv2
import os

video_path = "YOUR_VIDEO_NAME.mp4"   # put your uploaded video name here

os.makedirs("frames", exist_ok=True)

cap = cv2.VideoCapture(video_path)

count = 0
saved = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Save every 10th frame
    if count % 10 == 0:
        cv2.imwrite(f"frames/frame_{saved}.jpg", frame)
        saved += 1

    count += 1

cap.release()

print("Frames extracted:", saved)

Frames extracted: 0


In [10]:
import cv2

video_path = "1000182937.mp4"   # use exact filename

cap = cv2.VideoCapture(video_path)

print("Opened:", cap.isOpened())
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

cap.release()

Opened: True
Frames: 436


In [11]:
import cv2
import os

video_path = "1000182937.mp4"

os.makedirs("frames", exist_ok=True)

cap = cv2.VideoCapture(video_path)

count = 0
saved = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Save every 5th frame
    if count % 5 == 0:
        cv2.imwrite(f"frames/frame_{saved}.jpg", frame)
        saved += 1

    count += 1

cap.release()

print("Frames extracted:", saved)

Frames extracted: 88


In [12]:
!zip -r crowd_frames.zip frames

  adding: frames/ (stored 0%)
  adding: frames/frame_10.jpg (deflated 1%)
  adding: frames/frame_37.jpg (deflated 1%)
  adding: frames/frame_46.jpg (deflated 2%)
  adding: frames/frame_34.jpg (deflated 1%)
  adding: frames/frame_78.jpg (deflated 0%)
  adding: frames/frame_3.jpg (deflated 1%)
  adding: frames/frame_76.jpg (deflated 0%)
  adding: frames/frame_7.jpg (deflated 1%)
  adding: frames/frame_54.jpg (deflated 1%)
  adding: frames/frame_26.jpg (deflated 2%)
  adding: frames/frame_48.jpg (deflated 1%)
  adding: frames/frame_39.jpg (deflated 1%)
  adding: frames/frame_20.jpg (deflated 1%)
  adding: frames/frame_30.jpg (deflated 1%)
  adding: frames/frame_49.jpg (deflated 1%)
  adding: frames/frame_11.jpg (deflated 1%)
  adding: frames/frame_40.jpg (deflated 1%)
  adding: frames/frame_14.jpg (deflated 0%)
  adding: frames/frame_70.jpg (deflated 0%)
  adding: frames/frame_18.jpg (deflated 1%)
  adding: frames/frame_6.jpg (deflated 1%)
  adding: frames/frame_65.jpg (deflated 0%)
  add

In [13]:
from google.colab import files
files.download("crowd_frames.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from google.colab import files
files.download("crowd_frames.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import os

images = os.listdir("frames")
print("Total Images:", len(images))
print(images[:10])

Total Images: 88
['frame_10.jpg', 'frame_37.jpg', 'frame_46.jpg', 'frame_34.jpg', 'frame_78.jpg', 'frame_3.jpg', 'frame_76.jpg', 'frame_7.jpg', 'frame_54.jpg', 'frame_26.jpg']


In [16]:
!zip -r frames_only.zip frames

  adding: frames/ (stored 0%)
  adding: frames/frame_10.jpg (deflated 1%)
  adding: frames/frame_37.jpg (deflated 1%)
  adding: frames/frame_46.jpg (deflated 2%)
  adding: frames/frame_34.jpg (deflated 1%)
  adding: frames/frame_78.jpg (deflated 0%)
  adding: frames/frame_3.jpg (deflated 1%)
  adding: frames/frame_76.jpg (deflated 0%)
  adding: frames/frame_7.jpg (deflated 1%)
  adding: frames/frame_54.jpg (deflated 1%)
  adding: frames/frame_26.jpg (deflated 2%)
  adding: frames/frame_48.jpg (deflated 1%)
  adding: frames/frame_39.jpg (deflated 1%)
  adding: frames/frame_20.jpg (deflated 1%)
  adding: frames/frame_30.jpg (deflated 1%)
  adding: frames/frame_49.jpg (deflated 1%)
  adding: frames/frame_11.jpg (deflated 1%)
  adding: frames/frame_40.jpg (deflated 1%)
  adding: frames/frame_14.jpg (deflated 0%)
  adding: frames/frame_70.jpg (deflated 0%)
  adding: frames/frame_18.jpg (deflated 1%)
  adding: frames/frame_6.jpg (deflated 1%)
  adding: frames/frame_65.jpg (deflated 0%)
  add

In [17]:
from google.colab import files
files.download("frames_only.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# Install
!pip install -q ultralytics supervision opencv-python pandas

from ultralytics import YOLO
import supervision as sv
import cv2
import numpy as np
import pandas as pd
import os
from google.colab import files

# Upload Video
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Load Better Model
model = YOLO("yolov8x.pt")

# Open Video
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

# Output Video
out = cv2.VideoWriter(
    "crowd_panic_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# Tracker
tracker = sv.ByteTrack()

# History for stable movement calculation
previous_positions = {}
MAX_HISTORY = 5

os.makedirs("snapshots", exist_ok=True)

records = []
frame_no = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Person detection only
    results = model(
        frame,
        conf=0.25,
        iou=0.50,
        classes=[0],
        verbose=False
    )[0]

    detections = sv.Detections.from_ultralytics(results)

    detections = tracker.update_with_detections(detections)

    panic_count = 0
    warning_count = 0

    if detections.tracker_id is not None:

        for box, tid in zip(
            detections.xyxy,
            detections.tracker_id
        ):

            x1, y1, x2, y2 = box

            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            # Track history
            if tid not in previous_positions:
                previous_positions[tid] = []

            previous_positions[tid].append((cx, cy))

            if len(previous_positions[tid]) > MAX_HISTORY:
                previous_positions[tid].pop(0)

            speed = 0

            if len(previous_positions[tid]) >= 2:

                x_old, y_old = previous_positions[tid][0]
                x_new, y_new = previous_positions[tid][-1]

                speed = np.sqrt(
                    (x_new - x_old) ** 2 +
                    (y_new - y_old) ** 2
                )

            # Classification
            if speed < 5:

                state = "NORMAL"
                color = (0, 255, 0)

            elif speed < 12:

                state = "WARNING"
                color = (0, 255, 255)
                warning_count += 1

            else:

                state = "PANIC"
                color = (0, 0, 255)
                panic_count += 1

            # Draw box
            cv2.rectangle(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                color,
                2
            )

            # Label
            cv2.putText(
                frame,
                state,
                (int(x1), int(y1) - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

        # Cleanup old tracks
        if len(previous_positions) > 500:
            previous_positions.clear()

    people = len(detections)

    # Crowd Alert
    if panic_count >= 5:

        crowd_status = "PANIC ALERT"
        status_color = (0, 0, 255)

        cv2.imwrite(
            f"snapshots/panic_{frame_no}.jpg",
            frame
        )

    elif warning_count >= 5:

        crowd_status = "WARNING"
        status_color = (0, 255, 255)

    else:

        crowd_status = "NORMAL"
        status_color = (0, 255, 0)

    # Display Crowd Status
    cv2.putText(
        frame,
        crowd_status,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        status_color,
        3
    )

    cv2.putText(
        frame,
        f"People: {people}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    records.append([
        frame_no,
        people,
        warning_count,
        panic_count,
        crowd_status
    ])

    out.write(frame)

    frame_no += 1

cap.release()
out.release()

# Save Report
df = pd.DataFrame(
    records,
    columns=[
        "Frame",
        "People",
        "Warning_Count",
        "Panic_Count",
        "Status"
    ]
)

df.to_csv("crowd_report.csv", index=False)

print("DONE")
print("Video saved as crowd_panic_output.mp4")
print("CSV saved as crowd_report.csv")

Saving 1000182937.mp4 to 1000182937 (2).mp4
DONE
Video saved as crowd_panic_output.mp4
CSV saved as crowd_report.csv


In [21]:
from google.colab import files

files.download("crowd_panic_output.mp4")
files.download("crowd_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
!pip install -q ultralytics supervision opencv-python pandas

from ultralytics import YOLO
import supervision as sv
import cv2
import numpy as np
import pandas as pd
import os
from google.colab import files

# Upload video
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Load model
model = YOLO("yolov8x.pt")

# Open video
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

# Output video
out = cv2.VideoWriter(
    "crowd_panic_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

tracker = sv.ByteTrack()

previous_positions = {}
MAX_HISTORY = 5

records = []
frame_no = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(
        frame,
        conf=0.25,
        iou=0.5,
        classes=[0],
        verbose=False
    )[0]

    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)

    panic_count = 0
    warning_count = 0

    if detections.tracker_id is not None:

        for box, tid in zip(
            detections.xyxy,
            detections.tracker_id
        ):

            x1, y1, x2, y2 = box

            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            if tid not in previous_positions:
                previous_positions[tid] = []

            previous_positions[tid].append((cx, cy))

            if len(previous_positions[tid]) > MAX_HISTORY:
                previous_positions[tid].pop(0)

            speed = 0

            if len(previous_positions[tid]) >= 2:

                x_old, y_old = previous_positions[tid][0]
                x_new, y_new = previous_positions[tid][-1]

                speed = np.sqrt(
                    (x_new - x_old) ** 2 +
                    (y_new - y_old) ** 2
                )

            # More sensitive thresholds
            if speed < 2:
                color = (0, 255, 0)      # Green

            elif speed < 5:
                color = (0, 255, 255)    # Yellow
                warning_count += 1

            else:
                color = (0, 0, 255)      # Red
                panic_count += 1

            cv2.rectangle(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                color,
                2
            )

            # Show speed value instead of NORMAL/WARNING/PANIC text
            cv2.putText(
                frame,
                f"{speed:.1f}",
                (int(x1), int(y1)-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    people = len(detections)

    # Crowd status
    if panic_count >= 5:
        crowd_status = "PANIC ALERT"
        status_color = (0, 0, 255)

    elif warning_count >= 5:
        crowd_status = "WARNING"
        status_color = (0, 255, 255)

    else:
        crowd_status = "NORMAL"
        status_color = (0, 255, 0)

    cv2.putText(
        frame,
        crowd_status,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        status_color,
        3
    )

    cv2.putText(
        frame,
        f"People: {people}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,255,255),
        2
    )

    records.append([
        frame_no,
        people,
        warning_count,
        panic_count,
        crowd_status
    ])

    out.write(frame)

    frame_no += 1

cap.release()
out.release()

df = pd.DataFrame(
    records,
    columns=[
        "Frame",
        "People",
        "Warning_Count",
        "Panic_Count",
        "Status"
    ]
)

df.to_csv("crowd_report.csv", index=False)

print("DONE")
print("Video: crowd_panic_output.mp4")
print("CSV: crowd_report.csv")

Saving 1000182937.mp4 to 1000182937 (3).mp4
DONE
Video: crowd_panic_output.mp4
CSV: crowd_report.csv


In [23]:
from google.colab import files

files.download("crowd_panic_output.mp4")
files.download("crowd_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

print(video_path)

In [ ]:
video_path = list(uploaded.keys())[0]

In [24]:
!pip install -q ultralytics supervision opencv-python pandas

from ultralytics import YOLO
import supervision as sv
import cv2
import numpy as np
import pandas as pd
from google.colab import files

# Upload Video
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Load YOLOv8x
model = YOLO("yolov8x.pt")

# Open Video
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30

# Output Video
out = cv2.VideoWriter(
    "crowd_panic_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# Tracker
tracker = sv.ByteTrack()

previous_positions = {}
MAX_HISTORY = 10

records = []
frame_no = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(
        frame,
        conf=0.25,
        iou=0.5,
        classes=[0],   # person only
        verbose=False
    )[0]

    detections = sv.Detections.from_ultralytics(results)

    detections = tracker.update_with_detections(detections)

    panic_count = 0
    warning_count = 0

    if detections.tracker_id is not None:

        for box, tid in zip(
            detections.xyxy,
            detections.tracker_id
        ):

            x1, y1, x2, y2 = box

            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            if tid not in previous_positions:
                previous_positions[tid] = []

            previous_positions[tid].append((cx, cy))

            if len(previous_positions[tid]) > MAX_HISTORY:
                previous_positions[tid].pop(0)

            speed = 0

            if len(previous_positions[tid]) >= 2:

                x_old, y_old = previous_positions[tid][0]
                x_new, y_new = previous_positions[tid][-1]

                speed = np.sqrt(
                    (x_new - x_old) ** 2 +
                    (y_new - y_old) ** 2
                )

            # Classification
            if speed < 0.5:

                state = "NORMAL"
                color = (0,255,0)

            elif speed < 2:

                state = "WARNING"
                color = (0,255,255)
                warning_count += 1

            else:

                state = "PANIC"
                color = (0,0,255)
                panic_count += 1

            # Draw Box
            cv2.rectangle(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                color,
                2
            )

            # Show state + speed
            cv2.putText(
                frame,
                f"{state} {speed:.1f}",
                (int(x1), int(y1)-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2
            )

    people = len(detections)

    # Crowd Status
    if panic_count > 0:

        crowd_status = "PANIC ALERT"
        status_color = (0,0,255)

    elif warning_count > 0:

        crowd_status = "WARNING"
        status_color = (0,255,255)

    else:

        crowd_status = "NORMAL"
        status_color = (0,255,0)

    cv2.putText(
        frame,
        crowd_status,
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        status_color,
        3
    )

    cv2.putText(
        frame,
        f"People: {people}",
        (20,80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,255,255),
        2
    )

    records.append([
        frame_no,
        people,
        warning_count,
        panic_count,
        crowd_status
    ])

    out.write(frame)

    frame_no += 1

cap.release()
out.release()

# Save CSV
df = pd.DataFrame(
    records,
    columns=[
        "Frame",
        "People",
        "Warning_Count",
        "Panic_Count",
        "Status"
    ]
)

df.to_csv("crowd_report.csv", index=False)

print("DONE")
print("Video saved as crowd_panic_output.mp4")
print("CSV saved as crowd_report.csv")

Saving WhatsApp Video 2026-07-06 at 5.43.40 PM.mp4 to WhatsApp Video 2026-07-06 at 5.43.40 PM.mp4
DONE
Video saved as crowd_panic_output.mp4
CSV saved as crowd_report.csv


In [26]:
from google.colab import files

files.download("crowd_panic_output.mp4")
files.download("crowd_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
!apt-get -qq install ffmpeg

!ffmpeg -i crowd_panic_output.mp4 -vcodec libx264 final_output.mp4 -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [28]:
from google.colab import files
files.download("final_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
!apt-get -qq install ffmpeg -y

!ffmpeg -i crowd_panic_output.mp4 -c:v libx264 -c:a aac final_output.mp4 -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [30]:
from google.colab import files
files.download("final_output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
!ls -lh

total 235M
-rw-r--r-- 1 root root 2.5M Jul  9 15:33 '1000182937 (1).mp4'
-rw-r--r-- 1 root root 2.5M Jul  9 16:46 '1000182937 (2).mp4'
-rw-r--r-- 1 root root 2.5M Jul  9 16:51 '1000182937 (3).mp4'
-rw-r--r-- 1 root root 2.5M Jul  9 15:29  1000182937.mp4
-rw-r--r-- 1 root root  18M Jul  9 15:42  crowd_frames.zip
-rw-r--r-- 1 root root 6.6M Jul  9 17:01  crowd_panic_output.mp4
-rw-r--r-- 1 root root  19K Jul  9 17:01  crowd_report.csv
-rw-r--r-- 1 root root 2.0M Jul  9 17:11  final_output.mp4
drwxr-xr-x 2 root root 4.0K Jul  9 15:41  frames
-rw-r--r-- 1 root root  18M Jul  9 16:04  frames_only.zip
drwxr-xr-x 1 root root 4.0K Jun  4 13:39  sample_data
drwxr-xr-x 2 root root 4.0K Jul  9 15:33  snapshots
-rw-r--r-- 1 root root 1.6M Jul  9 17:00 'WhatsApp Video 2026-07-06 at 5.43.40 PM.mp4'
-rw-r--r-- 1 root root  50M Jul  9 15:30  yolov8m.pt
-rw-r--r-- 1 root root 131M Jul  9 16:46  yolov8x.pt


In [32]:
from IPython.display import Video
Video("crowd_panic_output.mp4", embed=True)

Output hidden; open in https://colab.research.google.com to view.

In [33]:
from IPython.display import FileLink

FileLink("crowd_panic_output.mp4")

/content/crowd_panic_output.mp4

In [34]:
from google.colab import files

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

print("New Video:", video_path)

Saving 1000182937.mp4 to 1000182937 (4).mp4
New Video: 1000182937 (4).mp4


In [35]:
!ffmpeg -i crowd_panic_output.mp4 -c:v libx264 -c:a aac final_output.mp4 -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [36]:
from IPython.display import FileLink

FileLink("final_output.mp4")

/content/final_output.mp4